In [229]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/datasets', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [230]:
import pandas as pd
import glob
import numpy as np
import ast
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
from sklearn.utils import resample
import tensorflow as tf
import csv
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout, TextVectorization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau

In [231]:
all_files = glob.glob("/content/drive/MyDrive/datasets/*.csv")

df = pd.concat([
    pd.read_csv(f, engine='python', quoting=csv.QUOTE_MINIMAL, on_bad_lines='skip')
    for f in all_files
], ignore_index=True)

print(f"Total rows: {len(df)}")
print(f"Dari {len(all_files)} file: {[os.path.basename(f) for f in all_files]}")
df.info()

Total rows: 12298
Dari 4 file: ['linkedin_jobs_20260505_063507.csv', 'linkedin_jobs_20260430_001046.csv', 'linkedin_jobs_20260426_002316.csv', 'jobstreet_final_20260509_173743.csv']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12298 entries, 0 to 12297
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                12095 non-null  float64
 1   title             12095 non-null  object 
 2   company           12095 non-null  object 
 3   location          12095 non-null  object 
 4   job_description   12263 non-null  object 
 5   job_url           12095 non-null  object 
 6   search_role       12298 non-null  object 
 7   search_location   12298 non-null  object 
 8   extracted_skills  12298 non-null  object 
 9   skills_count      12298 non-null  int64  
 10  scraped_at        12298 non-null  object 
 11  search_role_raw   4993 non-null   object 
 12  job_level         4993 non-null   object 
 13 

In [232]:
df = df.drop_duplicates()
df = df.drop(columns=['salary', 'search_role_raw', 'job_level', 'location_raw',
                       'salary_display', 'salary_min', 'salary_max', 'salary_avg'],
             errors='ignore')
print(f"Total duplicated rows: {df.duplicated().sum()}")
print(df.isnull().sum())

Total duplicated rows: 0
id                  203
title               203
company             203
location            203
job_description      35
job_url             203
search_role           0
search_location       0
extracted_skills      0
skills_count          0
scraped_at            0
dtype: int64


In [233]:
def parse_skills(val):
    try:
        result = ast.literal_eval(val)
        return ' '.join(result) if isinstance(result, list) and len(result) > 0 else ''
    except:
        return ''

df['extracted_skills_clean'] = df['extracted_skills'].apply(parse_skills)
print(f"Total rows: {len(df)}")
print(f"Skills kosong: {(df['extracted_skills_clean'] == '').sum()} rows")
print(df['search_role'].value_counts())

Total rows: 12298
Skills kosong: 44 rows
search_role
Software Engineer       1728
Data Engineer           1688
Backend Developer       1061
Frontend Developer      1043
Software                1000
Developer               1000
Full stack Developer    1000
IT                      1000
System Analyst           843
Data Analyst             636
IT Support               384
Data Scientist           246
Progammer                224
DevOps Engineer          202
Fullstack Developer      110
ML Engineer               83
Business Development      28
Other                     22
Name: count, dtype: int64


In [234]:
min_samples = 1000
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)
print(df['search_role'].value_counts())

search_role
Software Engineer       1728
Data Engineer           1688
Backend Developer       1061
Frontend Developer      1043
IT                      1000
Software                1000
Developer               1000
Full stack Developer    1000
Name: count, dtype: int64


In [235]:
df['text_input'] = df['job_description'] + ' ' + df['extracted_skills_clean']

df['search_role'] = df['search_role'].replace({'Full stack Developer': 'Fullstack Developer'})

roles_to_drop = ['Software Engineer', 'Web Developer', 'IT Support', 'IT',
                 'Developer', 'Software', 'System Analyst', 'Progammer', 'Other', 'Fullstack Developer']
df = df[~df['search_role'].isin(roles_to_drop)]

df['text_input'] = df['text_input'].fillna('').astype(str)
df = df[df['text_input'].str.strip() != ''].reset_index(drop=True)

min_samples = 200
role_counts = df['search_role'].value_counts()
valid_roles = role_counts[role_counts >= min_samples].index
df = df[df['search_role'].isin(valid_roles)].reset_index(drop=True)

print(df['search_role'].value_counts())

search_role
Data Engineer         1688
Backend Developer     1061
Frontend Developer    1042
Name: count, dtype: int64


In [236]:
encoder = LabelEncoder()
df['label'] = encoder.fit_transform(df['search_role'])
num_classes = len(encoder.classes_)
print(f"Total role unik: {num_classes}")
print(f"Mapping kelas:\n{dict(zip(encoder.classes_, range(num_classes)))}")

target_samples = 500
dfs_resampled = []
for role in df['search_role'].unique():
    df_role = df[df['search_role'] == role]
    if len(df_role) < target_samples:
        df_role = resample(df_role, replace=True, n_samples=target_samples, random_state=42)
    else:
        df_role = df_role.sample(n=target_samples, random_state=42)
    dfs_resampled.append(df_role)

df = pd.concat(dfs_resampled).sample(frac=1, random_state=42).reset_index(drop=True)
df['label'] = encoder.transform(df['search_role'])
print(df['search_role'].value_counts())

Total role unik: 3
Mapping kelas:
{'Backend Developer': 0, 'Data Engineer': 1, 'Frontend Developer': 2}
search_role
Frontend Developer    500
Data Engineer         500
Backend Developer     500
Name: count, dtype: int64


In [237]:
X_train, X_val, y_train, y_val = train_test_split(
    df['text_input'], df['label'],
    test_size=0.2, random_state=42, stratify=df['label']
)

vectorizer = TextVectorization(
    max_tokens=20000,
    output_mode='int',
    output_sequence_length=512
)
vectorizer.adapt(X_train.to_numpy())
print("Shape X_train:", X_train.shape)

Shape X_train: (1200,)


In [238]:
class CustomEarlyStopping(tf.keras.callbacks.Callback):
    def __init__(self, patience=5):
        super().__init__()
        self.patience = patience
        self.best_weights = None
        self.best_loss = np.inf
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        current_loss = logs.get('val_loss')
        if current_loss < self.best_loss:
            self.best_loss = current_loss
            self.wait = 0
            self.best_weights = self.model.get_weights()
        else:
            self.wait += 1
            if self.wait >= self.patience:
                print(f"\n[INFO] val_loss tidak membaik selama {self.patience} epoch. Stop training!")
                self.model.stop_training = True
                print("[INFO] Mengembalikan bobot model ke epoch terbaik.")
                self.model.set_weights(self.best_weights)

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    CustomEarlyStopping(patience=5)
]

In [239]:
vocab_size = len(vectorizer.get_vocabulary())

inputs = Input(shape=(1,), dtype=tf.string, name='text_input')
x = vectorizer(inputs)
x = Embedding(input_dim=vocab_size, output_dim=64)(x)
x = GlobalAveragePooling1D()(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
outputs = Dense(num_classes, activation='softmax', name='role_output')(x)

model = Model(inputs=inputs, outputs=outputs, name="Job_Role_Classifier")
model.compile(
    optimizer=Adam(learning_rate=0.002),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

print("Mulai proses training...")
history = model.fit(
    X_train.to_numpy(), y_train,
    validation_data=(X_val.to_numpy(), y_val),
    epochs=40,
    batch_size=64,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

Mulai proses training...
Epoch 1/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 43ms/step - accuracy: 0.3942 - loss: 1.0804 - val_accuracy: 0.4167 - val_loss: 1.0539 - learning_rate: 0.0020
Epoch 2/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.4458 - loss: 1.0545 - val_accuracy: 0.4733 - val_loss: 1.0217 - learning_rate: 0.0020
Epoch 3/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4558 - loss: 1.0331 - val_accuracy: 0.4900 - val_loss: 1.0086 - learning_rate: 0.0020
Epoch 4/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4692 - loss: 1.0117 - val_accuracy: 0.4533 - val_loss: 1.0037 - learning_rate: 0.0020
Epoch 5/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5108 - loss: 0.9885 - val_accuracy: 0.4667 - val_loss: 0.9971 - learning_rate: 0.0020
Epoch 6/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.5592 - loss: 0.9419 - val_accuracy: 0.5600 - val_loss: 0.9393 - learning_rate: 0.0020
Epoch 7/40
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6125

In [240]:
y_pred = np.argmax(model.predict(X_val.to_numpy()), axis=1)
print(classification_report(y_val, y_pred, target_names=encoder.classes_))

10/10 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step
                    precision    recall  f1-score   support

 Backend Developer       0.53      0.47      0.50       100
     Data Engineer       0.58      0.57      0.58       100
Frontend Developer       0.71      0.80      0.75       100

          accuracy                           0.61       300
         macro avg       0.61      0.61      0.61       300
      weighted avg       0.61      0.61      0.61       300



In [241]:
model.save('job_role_model.keras')
print("[INFO] Model berhasil disimpan")

loaded_model = tf.keras.models.load_model('job_role_model.keras')

def rekomendasi_role(teks_skill_user):
    input_tensor = tf.constant([teks_skill_user], dtype=tf.string)
    pred_probs = loaded_model.predict(input_tensor, verbose=0)
    pred_class_idx = np.argmax(pred_probs)
    predicted_role = encoder.inverse_transform([pred_class_idx])[0]
    return predicted_role

skill_input = "i have experience for creating web interface using reactjs, next.js, tailwind, and using operating system linux ubuntu"
hasil_prediksi = rekomendasi_role(skill_input)

print("\n--- Hasil Inference ---")
print(f"Skill User : {skill_input}")
print(f"Cocok untuk: {hasil_prediksi}")

[INFO] Model berhasil disimpan

--- Hasil Inference ---
Skill User : i have experience for creating web interface using reactjs, next.js, tailwind, and using operating system linux ubuntu
Cocok untuk: Frontend Developer
